# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from utils import load_pickle

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

/home/junior/Documentos/GitHub/kaggle-competition-predicting-stellar-class/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
label_encoder = load_pickle('../models/label_encoder.pkl')

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/X_train_stacking_layer_two.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_two.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.999980,0.000019,7.173664e-07,0.999940,0.000059,0.000001,0.999902,0.000087,0.000011,0.999964,0.000035,9.391767e-07,9.999364e-01,0.000064,0.000000,0.998533,0.001276,0.000191
1,0.993270,0.000283,6.446509e-03,0.994368,0.000245,0.005387,0.993982,0.000245,0.005772,0.992230,0.000265,7.504966e-03,9.951892e-01,0.000171,0.004639,0.972023,0.002391,0.025586
2,0.000034,0.999962,4.381048e-06,0.000139,0.999859,0.000002,0.000214,0.999774,0.000012,0.000013,0.999981,6.276192e-06,3.944814e-07,1.000000,0.000000,0.000114,0.999862,0.000024
3,0.999879,0.000120,7.617780e-07,0.999812,0.000187,0.000001,0.999843,0.000145,0.000012,0.999931,0.000069,5.789762e-07,9.998199e-01,0.000180,0.000000,0.998349,0.001455,0.000196
4,0.997913,0.002069,1.828613e-05,0.998790,0.001202,0.000008,0.998263,0.001704,0.000033,0.996330,0.003666,3.709336e-06,9.977873e-01,0.002205,0.000008,0.994559,0.004657,0.000784


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.998106,0.001851,0.000042,0.997287,0.002522,0.000192,0.998122,0.001845,0.000033,0.993568,0.006380,0.000053,0.998112,0.001858,0.000030,0.991100,0.006021,0.002879
1,0.997635,0.002352,0.000014,0.997773,0.002224,0.000003,0.997897,0.002071,0.000032,0.996850,0.003147,0.000003,0.998374,0.001623,0.000003,0.993814,0.005762,0.000424
2,0.997136,0.001323,0.001541,0.997897,0.000212,0.001892,0.997850,0.000528,0.001622,0.992689,0.000960,0.006351,0.997662,0.000721,0.001618,0.987143,0.003596,0.009262
3,0.000485,0.000066,0.999449,0.001142,0.000192,0.998666,0.000653,0.000081,0.999266,0.000062,0.000010,0.999928,0.000413,0.000015,0.999572,0.000552,0.000322,0.999126
4,0.999485,0.000510,0.000004,0.999636,0.000360,0.000003,0.999666,0.000319,0.000016,0.999455,0.000531,0.000014,0.999776,0.000222,0.000002,0.998133,0.001603,0.000264


# Machine Learning

In [8]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train_fold = X.iloc[train_idx, :]
        X_valid_fold = X.iloc[valid_idx, :]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        solver = trial.suggest_categorical("solver", ["svd", "lsqr", "eigen"])
        shrinkage = None
        tol = 0.0001

        if solver in ["lsqr", "eigen"]:
            shrinkage = trial.suggest_float("shrinkage", 0.0, 1.0)

        if solver == "svd":
            tol = trial.suggest_float("tol", 1e-5, 1e-3, log=True)

        params = {
            "solver": solver,
            "shrinkage": shrinkage,
            "tol": tol
        }
        
        model = LinearDiscriminantAnalysis(**params).fit(X_train_fold, y_train_fold)

        proba = model.predict_proba(X_valid_fold)

        w0 = trial.suggest_float('weight_class_0', 0.1, 100.0)
        w1 = trial.suggest_float('weight_class_1', 0.1, 100.0)
        w2 = trial.suggest_float('weight_class_2', 0.1, 100.0)

        weights = np.array([w0, w1, w2])
        weighted_probas = proba * weights

        pred = np.argmax(weighted_probas, axis=1)
        
        score = balanced_accuracy_score(y_valid_fold, pred)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=120, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-06-16 17:33:00,867] A new study created in memory with name: no-name-7a7a0716-9c09-4e7e-aa64-49e772ccce90
Best trial: 0. Best value: 0.961828:   1%|█▏                                                                                                                                        | 1/120 [00:08<16:08,  8.14s/it]

[I 2026-06-16 17:33:08,986] Trial 0 finished with value: 0.961828181048854 and parameters: {'solver': 'lsqr', 'shrinkage': 0.6211592122685499, 'weight_class_0': 39.02841183483406, 'weight_class_1': 7.179990423571741, 'weight_class_2': 41.89119129334394}. Best is trial 0 with value: 0.961828181048854.


[I 2026-06-16 17:33:10,197] Trial 11 finished with value: 0.9589576621537717 and parameters: {'solver': 'lsqr', 'shrinkage': 0.04148010460643481, 'weight_class_0': 23.028866441111138, 'weight_class_1': 22.657759345841207, 'weight_class_2': 57.29762723706113}. Best is trial 0 with value: 0.961828181048854.


[I 2026-06-16 17:33:10,264] Trial 5 finished with value: 0.9593818575874898 and parameters: {'solver': 'eigen', 'shrinkage': 0.04487390966375282, 'weight_class_0': 21.291931343338362, 'weight_class_1': 68.79794091132335, 'weight_class_2': 93.08898109250971}. Best is trial 0 with value: 0.961828181048854.
[I 2026-06-16 17:33:10,294] Trial 4 finished with value: 0.9624840622580102 and parameters: {'solver': 'eigen', 'shrinkage': 0.7701498108570852, 'weight_class_0': 69.61731779480208, 'weight_class_1': 37.20150065515457, 'weight_class_2': 64.30012255578146}. Best is trial 4 with value: 0.9624840622580102.
[I 2026-06-16 17:33:10,304] Trial 1 finished with value: 0.9584652157831108 and parameters: {'solver': 'lsqr', 'shrinkage': 0.05999369396652576, 'weight_class_0': 63.480249862711695, 'weight_class_1': 15.513311100474095, 'weight_class_2': 54.0812224009997}. Best is trial 4 with value: 0.9624840622580102.
[I 2026-06-16 17:33:10,568] Trial 9 pruned. 


Best trial: 4. Best value: 0.962484:   8%|██████████▎                                                                                                                               | 9/120 [00:10<00:59,  1.85it/s]

...

[I 2026-06-16 17:34:03,910] Trial 116 finished with value: 0.9631744819202481 and parameters: {'solver': 'lsqr', 'shrinkage': 0.8858925310799832, 'weight_class_0': 1.0883971522790499, 'weight_class_1': 0.20534227315229714, 'weight_class_2': 96.50583232353307}. Best is trial 110 with value: 0.9635581897489761.


Best trial: 110. Best value: 0.963558:  98%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊  | 118/120 [01:03<00:00,  3.31it/s]

[I 2026-06-16 17:34:04,250] Trial 117 finished with value: 0.9635359419252927 and parameters: {'solver': 'lsqr', 'shrinkage': 0.8836404689853112, 'weight_class_0': 0.189629138145367, 'weight_class_1': 66.74251351601525, 'weight_class_2': 86.26385364245772}. Best is trial 110 with value: 0.9635581897489761.
[I 2026-06-16 17:34:04,451] Trial 119 finished with value: 0.9634876247517801 and parameters: {'solver': 'eigen', 'shrinkage': 0.9307526979744185, 'weight_class_0': 0.3269547256106984, 'weight_class_1': 34.629705029148994, 'weight_class_2': 87.89806419225596}. Best is trial 110 with value: 0.9635581897489761.


Best trial: 110. Best value: 0.963558: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 120/120 [01:03<00:00,  1.89it/s]

[I 2026-06-16 17:34:04,458] Trial 118 finished with value: 0.9635243818357366 and parameters: {'solver': 'lsqr', 'shrinkage': 0.9333429374411465, 'weight_class_0': 0.20455783452251236, 'weight_class_1': 30.833369036484832, 'weight_class_2': 74.77551594417292}. Best is trial 110 with value: 0.9635581897489761.
Best trial score:
0.9635581897489761

Best params:
{'solver': 'lsqr', 'shrinkage': 0.8816069104818597, 'weight_class_0': 0.13032685601422409, 'weight_class_1': 60.29193495547739, 'weight_class_2': 85.9716417473798}


In [14]:
lda_params = {k: v for k, v in study.best_params.items() if k not in ['weight_class_0', 'weight_class_1', 'weight_class_2']}

model = LinearDiscriminantAnalysis(**lda_params).fit(X_train, y_train.class_encoded)

test_proba = model.predict_proba(X_test)

weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = test_proba * weights

pred = np.argmax(weighted_probas, axis=1)

In [15]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [16]:
submission = pd.read_csv('../data/sample_submission.csv')
submission['class'] = sub_labels

submission.to_csv('../data/submission_stacking_lda.csv', index=False)

In [17]:
submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [18]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'lgbm_2', 'cat_0', 'cat_1', 'cat_2', 'xgb_0',
       'xgb_1', 'xgb_2', 'hist_0', 'hist_1', 'hist_2', 'rf_0', 'rf_1', 'rf_2',
       'extra_0', 'extra_1', 'extra_2'],
      dtype='str')